In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
"""
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
"""
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

"\nimport os\nfor dirname, _, filenames in os.walk('/kaggle/input'):\n    for filename in filenames:\n        print(os.path.join(dirname, filename))\n"

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
import numpy as np
import pandas as pd
import os
from glob import glob
from sklearn.metrics import accuracy_score
from PIL import Image, ImageEnhance, ImageFilter
import torchvision.transforms as transforms
import random

# ============================================
# DATA LOADING FROM FLAT FOLDER
# ============================================

def load_kids_from_flat_folder(folder_path):
    """
    Load all kids data from a flat folder structure
    Extracts kid ID from filename (e.g., F001_xxx.jpg -> F001)
    """
    data = []
    
    # Get all image files
    image_files = glob(os.path.join(folder_path, "*.*"))
    image_files = [f for f in image_files 
                  if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    print(f"Found {len(image_files)} images in {folder_path}")
    
    # Extract kid IDs from filenames
    for image_path in image_files:
        filename = os.path.basename(image_path)
        
        # Extract kid ID from filename
        # Assumes format like: F001_xxx.jpg, F020_xxx.jpg, etc.
        kid_id = filename.split('_')[0]  # Gets 'F001' from 'F001_xxx.jpg'
        
        data.append({
            'ID': kid_id,
            'Path': image_path
        })
    
    df = pd.DataFrame(data)
    
    # Count kids and images per kid
    kid_counts = df['ID'].value_counts()
    print(f"\nFound {len(kid_counts)} unique kids:")
    for kid_id, count in kid_counts.items():
        print(f"  {kid_id}: {count} images")
    
    print(f"\nTotal: {len(df)} images from {len(kid_counts)} kids")
    return df




In [35]:
# ============================================
# AGGRESSIVE AUGMENTATION TRANSFORMS
# ============================================

class AggressiveAugmentation:
    """
    VERY AGGRESSIVE augmentation to create maximum variation
    from limited training data
    """
    def __init__(self, img_size=96):
        self.img_size = img_size
        
    def __call__(self, img):
        # First resize to slightly larger size
        img = transforms.Resize((int(self.img_size * 1.15), int(self.img_size * 1.15)))(img)
        
        # Random crop to target size (creates scale variation)
        img = transforms.RandomCrop(self.img_size)(img)
        
        # Random horizontal flip
        if random.random() > 0.5:
            img = transforms.functional.hflip(img)
        
        # Random rotation (-25 to +25 degrees)
        angle = random.uniform(-25, 25)
        img = transforms.functional.rotate(img, angle)
        
        # Aggressive color jittering
        img = transforms.ColorJitter(
            brightness=0.4,  # Very aggressive
            contrast=0.4,
            saturation=0.4,
            hue=0.15
        )(img)
        
        # Random blur or sharpen
        if random.random() > 0.7:
            if random.random() > 0.5:
                img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.5)))
            else:
                enhancer = ImageEnhance.Sharpness(img)
                img = enhancer.enhance(random.uniform(1.2, 2.0))
        
        # Random noise
        if random.random() > 0.8:
            img_array = np.array(img)
            noise = np.random.normal(0, 10, img_array.shape)
            img_array = np.clip(img_array + noise, 0, 255).astype(np.uint8)
            img = Image.fromarray(img_array)
        
        # Random brightness adjustment
        if random.random() > 0.5:
            enhancer = ImageEnhance.Brightness(img)
            img = enhancer.enhance(random.uniform(0.7, 1.3))
        
        # Convert to tensor and normalize
        img = transforms.ToTensor()(img)
        img = transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                  std=[0.229, 0.224, 0.225])(img)
        
        return img

def get_test_transform(img_size=96):
    """Simple transform for testing (no augmentation)"""
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])

In [36]:
# ============================================
# OPTIMIZED MODEL - SIMPLER ARCHITECTURE
# ============================================

class FastFaceRecognitionCNN(nn.Module):
    def __init__(self, num_classes):
        super(FastFaceRecognitionCNN, self).__init__()
        
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),
            
            # Block 2
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.3),
            
            # Block 3
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.4),
            
            # Global Average Pooling
            nn.AdaptiveAvgPool2d((1, 1))
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

class KidsFaceDataset(Dataset):
    def __init__(self, kids_dataframe, transform=None):
        self.kids_data = kids_dataframe.reset_index(drop=True)
        self.transform = transform
        
        self.unique_ids = sorted(self.kids_data['ID'].unique())
        self.id_to_label = {kid_id: idx for idx, kid_id in enumerate(self.unique_ids)}
        self.label_to_id = {idx: kid_id for kid_id, idx in self.id_to_label.items()}
        
        print(f"\n📋 Label Mapping:")
        for kid_id, label in sorted(self.id_to_label.items()):
            print(f"  {kid_id} → Label {label}")
        
    def __len__(self):
        return len(self.kids_data)
    
    def __getitem__(self, idx):
        kid_row = self.kids_data.iloc[idx]
        image_path = kid_row['Path']
        kid_id = kid_row['ID']
        
        try:
            image = Image.open(image_path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")
            image = Image.new('RGB', (96, 96), color='black')
        
        if self.transform:
            image = self.transform(image)
            
        label = self.id_to_label[kid_id]
        return image, label
    
    def get_id_from_label(self, label):
        return self.label_to_id[label]

In [ ]:
def train_with_augmentation(dataset, num_epochs=200, batch_size=20):
    """
    Train with AGGRESSIVE augmentation
    Each epoch sees different augmented versions of the same images
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    num_classes = len(dataset.unique_ids)
    
    print("=" * 70)
    print("TRAINING WITH AGGRESSIVE AUGMENTATION")
    print("=" * 70)
    print(f"Number of kids: {num_classes}")
    print(f"Training images: {len(dataset)}")
    print(f"Epochs: {num_epochs}")
    print(f"Batch size: {batch_size}")
    print(f"Device: {device}")
    print(f"\n🔥 Each epoch creates NEW augmented variations!")
    print(f"   Total training variations: {num_epochs} × {len(dataset)} = {num_epochs * len(dataset)}")
    
    # Create training dataset with AGGRESSIVE augmentation
    train_dataset = KidsFaceDataset(dataset.kids_data, 
                                    transform=AggressiveAugmentation(img_size=96))
    
    train_loader = DataLoader(train_dataset, 
                             batch_size=batch_size, 
                             shuffle=True,
                             num_workers=0)  # Set to 0 for Windows compatibility
    
    # Initialize model
    model = FastFaceRecognitionCNN(num_classes=num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=10, factor=0.5)
    criterion = nn.CrossEntropyLoss()
    
    # Training
    print("\n🏃 Training with aggressive augmentation...")
    best_acc = 0
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        epoch_acc = 100 * correct / total
        avg_loss = epoch_loss / len(train_loader)
        scheduler.step(avg_loss)
        
        if epoch_acc > best_acc:
            best_acc = epoch_acc
            torch.save(model.state_dict(), 'best_face_model.pth')
        
        if (epoch + 1) % 10 == 0:
            print(f"   Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f} - Acc: {epoch_acc:.2f}% - Best: {best_acc:.2f}%")
    
    # Load best model
    model.load_state_dict(torch.load('best_face_model.pth'))
    print(f"\n Best model saved (Training Acc: {best_acc:.2f}%)")
    
    return model

In [ ]:
def test_on_all_images(model, dataset):
    """Test the trained model on all images (without augmentation)"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    
    print("\n" + "=" * 70)
    print("TESTING ON ALL IMAGES (No Augmentation)")
    print("=" * 70)
    
    # Create test dataset WITHOUT augmentation
    test_dataset = KidsFaceDataset(dataset.kids_data, 
                                   transform=get_test_transform(img_size=96))
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    
    all_predictions = []
    all_true_labels = []
    correct_per_kid = {kid_id: {'correct': 0, 'total': 0} 
                       for kid_id in dataset.unique_ids}
    
    with torch.no_grad():
        for idx, (images, labels) in enumerate(test_loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            pred_label = predicted.cpu().numpy()[0]
            true_label = labels.cpu().numpy()[0]
            
            all_predictions.append(pred_label)
            all_true_labels.append(true_label)
            
            true_id = dataset.get_id_from_label(true_label)
            pred_id = dataset.get_id_from_label(pred_label)
            
            correct_per_kid[true_id]['total'] += 1
            if pred_label == true_label:
                correct_per_kid[true_id]['correct'] += 1
    
    overall_accuracy = accuracy_score(all_true_labels, all_predictions)
    
    print("\n Results per Kid:")
    for kid_id in sorted(correct_per_kid.keys()):
        stats = correct_per_kid[kid_id]
        acc = 100 * stats['correct'] / stats['total'] if stats['total'] > 0 else 0
        status = "warning" if acc == 100 else "true" if acc >= 50 else "false"
        print(f"  {status} {kid_id}: {stats['correct']}/{stats['total']} correct ({acc:.1f}%)")
    
    print(f"\n🎯 Overall Test Accuracy: {overall_accuracy * 100:.2f}%")
    
    return all_true_labels, all_predictions, overall_accuracy


In [ ]:
# ============================================
# MAIN EXECUTION
# ============================================

def main():
    print(" FACE RECOGNITION WITH AGGRESSIVE AUGMENTATION")
    print(" Loading data from folder...")
    
    folder_path = r"C:\users\omar_\Downloads\Kid_face_recognetion\Face_recognition\Trained_image"
    all_kids = load_kids_from_flat_folder(folder_path)
    
    if len(all_kids) == 0:
        print(" No images found! Please check the folder path.")
        return
    
    # Create dataset
    dataset = KidsFaceDataset(all_kids, transform=None)
    
    print(f"\n Dataset created:")
    print(f"   Total images: {len(dataset)}")
    print(f"   Unique kids: {len(dataset.unique_ids)}")
    
    # Train with aggressive augmentation
    model = train_with_augmentation(dataset, num_epochs=200, batch_size=20)
    
    # Test on original images
    true_labels, predictions, accuracy = test_on_all_images(model, dataset)
    
    print("\n Training and testing completed!")
    print(f" Final Test Accuracy: {accuracy * 100:.2f}%")

if __name__ == "__main__":
    main()

🚀 FACE RECOGNITION WITH AGGRESSIVE AUGMENTATION
📂 Loading data from folder...
Found 20 images in C:\users\omar_\Downloads\Kid_face_recognetion\Face_recognition\Trained_image

Found 20 unique kids:
  F001: 1 images
  F002: 1 images
  F003: 1 images
  F004: 1 images
  F005: 1 images
  F006: 1 images
  F007: 1 images
  F008: 1 images
  F009: 1 images
  F010: 1 images
  F011: 1 images
  F012: 1 images
  F013: 1 images
  F014: 1 images
  F015: 1 images
  F016: 1 images
  F017: 1 images
  F018: 1 images
  F019: 1 images
  F020: 1 images

Total: 20 images from 20 kids

📋 Label Mapping:
  F001 → Label 0
  F002 → Label 1
  F003 → Label 2
  F004 → Label 3
  F005 → Label 4
  F006 → Label 5
  F007 → Label 6
  F008 → Label 7
  F009 → Label 8
  F010 → Label 9
  F011 → Label 10
  F012 → Label 11
  F013 → Label 12
  F014 → Label 13
  F015 → Label 14
  F016 → Label 15
  F017 → Label 16
  F018 → Label 17
  F019 → Label 18
  F020 → Label 19

✅ Dataset created:
   Total images: 20
   Unique kids: 20
TRAIN

In [ ]:
# ============================================
# EXTRACT EMBEDDINGS FROM TRAINED MODEL
# ============================================

def extract_embeddings(model, dataset, device='cpu'):
    """Extract embeddings for all kids"""
    model.to(device)
    model.eval()
    
    print("\n" + "=" * 70)
    print("EXTRACTING EMBEDDINGS FOR ALL KIDS")
    print("=" * 70)
    
    test_transform = transforms.Compose([
        transforms.Resize((96, 96)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    embeddings_dict = {}
    
    with torch.no_grad():
        for idx in range(len(dataset)):
            kid_row = dataset.kids_data.iloc[idx]
            image_path = kid_row['Path']
            kid_id = kid_row['ID']
            
            image = Image.open(image_path).convert('RGB')
            image_tensor = test_transform(image).unsqueeze(0).to(device)
            
            # Extract features (embedding)
            features = model.features(image_tensor)
            embedding = features.view(features.size(0), -1)
            
            embeddings_dict[kid_id] = {
                'embedding': embedding.cpu().numpy(),
                'image_path': image_path,
                'label': dataset.id_to_label[kid_id]
            }
            
            print(f"   {kid_id}: Embedding shape {embedding.shape}")
    
    print(f"\nExtracted embeddings for {len(embeddings_dict)} kids")
    return embeddings_dict

def save_model_and_embeddings(model, embeddings_dict, dataset):
    """Save model, embeddings, and metadata"""
    import pickle
    
    print("\n" + "=" * 70)
    print("SAVING MODEL AND EMBEDDINGS")
    print("=" * 70)
    
    # Save model
    torch.save(model.state_dict(), 'face_recognition_model.pth')
    print(" Model saved to: face_recognition_model.pth")
    
    # Save embeddings
    with open('face_embeddings.pkl', 'wb') as f:
        pickle.dump(embeddings_dict, f)
    print(" Embeddings saved to: face_embeddings.pkl")
    
    # Save metadata
    metadata = {
        'id_to_label': dataset.id_to_label,
        'label_to_id': dataset.label_to_id,
        'unique_ids': dataset.unique_ids,
        'num_classes': len(dataset.unique_ids)
    }
    with open('face_metadata.pkl', 'wb') as f:
        pickle.dump(metadata, f)
    print("Metadata saved to: face_metadata.pkl")
    
    print("\n All files saved successfully!")

In [48]:
# Extract and save embeddings

folder_path = r"C:\users\omar_\Downloads\Kid_face_recognetion\Face_recognition\Trained_image"
all_kids = load_kids_from_flat_folder(folder_path)
dataset = KidsFaceDataset(all_kids, transform=None)
model = train_with_augmentation(dataset, num_epochs=200, batch_size=20)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
embeddings_dict = extract_embeddings(model, dataset, device)
save_model_and_embeddings(model, embeddings_dict, dataset)

Found 20 images in C:\users\omar_\Downloads\Kid_face_recognetion\Face_recognition\Trained_image

Found 20 unique kids:
  F001: 1 images
  F002: 1 images
  F003: 1 images
  F004: 1 images
  F005: 1 images
  F006: 1 images
  F007: 1 images
  F008: 1 images
  F009: 1 images
  F010: 1 images
  F011: 1 images
  F012: 1 images
  F013: 1 images
  F014: 1 images
  F015: 1 images
  F016: 1 images
  F017: 1 images
  F018: 1 images
  F019: 1 images
  F020: 1 images

Total: 20 images from 20 kids

📋 Label Mapping:
  F001 → Label 0
  F002 → Label 1
  F003 → Label 2
  F004 → Label 3
  F005 → Label 4
  F006 → Label 5
  F007 → Label 6
  F008 → Label 7
  F009 → Label 8
  F010 → Label 9
  F011 → Label 10
  F012 → Label 11
  F013 → Label 12
  F014 → Label 13
  F015 → Label 14
  F016 → Label 15
  F017 → Label 16
  F018 → Label 17
  F019 → Label 18
  F020 → Label 19
TRAINING WITH AGGRESSIVE AUGMENTATION
Number of kids: 20
Training images: 20
Epochs: 200
Batch size: 20
Device: cpu

🔥 Each epoch creates NEW 